Notes(What needs to be done): 
1. check for 
2. 
etc

markdown
# 01 - Data Checks

**Purpose:** Establish what the raw data actually means before any cleaning or
modeling. The case document states policy limits but is silent on deductibles. This notebook resolves each
from the data where possible, and records an explicit assumption where not.

**Input:** `data/raw/CASComp Main DataSet.csv` (unmodified)
**Output:** No files. Conclusions only, carried into 02_Cleaned_Data.
**Depends on:** 00_Inventory

**Questions addressed**

1. Is there a deductible?
2. Are losses censored at the policy limits?
3. What does one row represent?
4. Are claim counts consistent with a Poisson process?
5. What period does the data cover, and are claims developed?
6. What is the exposure per record?
7. Is `risk_tier` a function of variables already in the dataset?


## 1. Initial Histograms
plot histograms to show claim frequencies 

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
df = pd.read_csv(RAW / "CASComp Main DataSet.csv")

In [ ]:
counts = (df.assign(is_claim=(df.claim_id > 0).astype(int))
            .groupby(['student_id', 'coverage'], as_index=False)
            .is_claim.sum()
            .rename(columns={'is_claim': 'claim_count'}))

counts.groupby('coverage').claim_count.value_counts().unstack(fill_value=0)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (cov, g) in zip(axes.flat, counts.groupby('coverage')):
    g.claim_count.value_counts().sort_index().plot.bar(ax=ax, rot=0)
    ax.set_title(cov)
    ax.set_xlabel('claims per student')
    ax.set_ylabel('students')
plt.tight_layout()

In [ ]:
pos = df[df.amount > 0]

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (cov, g) in zip(axes.flat, pos.groupby('coverage')):
    g.amount.hist(bins=50, ax=ax)
    ax.set_title(f"{cov}  (n={len(g)})")
    ax.set_xlabel('claim amount')
    ax.set_ylabel('claims')
plt.tight_layout()

**Finding:** Not much can be said about the claim frequency histogram at this moment. The Claim severity histograms do not show any obvious signs of a deductible but further analysis of the lower tails will make the image clearer

## 2. Deductible Analysis
Conclude whether or not there is a deductible

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (cov, g) in zip(axes.flat, pos.groupby('coverage')):
    g[g.amount < 1500].amount.hist(bins=60, ax=ax)
    ax.set_title(cov)
    ax.set_xlabel('claim amount')
plt.tight_layout()

Finding:** We can not see evidence of a deductible so we will move foward under the assumption that there is no deductible and that the policy limits are consistent with the ones stated